# Semana 9 — Estimación de Fase y Algoritmo de Shor
## La amenaza a RSA y la aceleración exponencial

**Objetivo:** Entender la Estimación de Fase Cuántica (QPE), su conexión con la QFT, y cómo Shor reduce la factorización a encontrar el período de una función modular.

**Hito verificable:** Implementas QPE completo y factorizas N=15 usando el algoritmo de Shor (versión simplificada con oráculo hardcodeado).

In [ ]:
import numpy as np
from math import gcd
from fractions import Fraction
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector
import matplotlib.pyplot as plt
print('Librerías importadas correctamente')

## Ejercicio 9.1 — Estimación de Fase Cuántica (QPE)
Dado U|u⟩ = e^(2πiφ)|u⟩, QPE estima φ con n bits de precisión.

In [ ]:
def qft_matrix(n):
    N = 2**n
    omega = np.exp(2j * np.pi / N)
    j, k = np.meshgrid(np.arange(N), np.arange(N))
    return omega**(j*k) / np.sqrt(N)

def qpe_matrix(U, eigenstate, n_ancilla):
    """
    QPE matricial: estima la fase del eigenvalue e^(2πiφ) de U.
    Devuelve el histograma de probabilidades de los estados ancilla.
    """
    N_anc = 2**n_ancilla
    dim_U = len(eigenstate)
    
    # Estado inicial: |0...0⟩ ⊗ |u⟩
    anc = np.zeros(N_anc, dtype=complex)
    anc[0] = 1.0
    psi = np.kron(anc, eigenstate)
    
    # Hadamard en todos los ancilla
    H_anc = np.ones((N_anc, N_anc), dtype=complex) / np.sqrt(N_anc)
    psi = np.kron(H_anc @ np.eye(N_anc, dtype=complex), np.eye(dim_U, dtype=complex)) @ psi
    # Simplificado: el estado de los ancilla tras las H es superposición uniforme
    psi_anc = np.ones(N_anc, dtype=complex) / np.sqrt(N_anc)
    
    # Aplicar U^(2^k) controlado: cada ancilla k acumula fase 2^k · φ
    # La fase relativa entre |0⟩ y |1⟩ del ancilla k es e^(2πi·2^k·φ)
    # Tras todas las U controladas, el estado ancilla es:
    # (1/√N) Σ_j e^(2πi·j·φ) |j⟩
    phase = np.angle(np.linalg.eigvals(U)[np.argmax(np.abs(eigenstate))])
    phi = phase / (2 * np.pi) % 1  # Fase en [0,1)
    
    # Amplitudes antes de la IQFT
    psi_pre_iqft = np.array([np.exp(2j*np.pi*j*phi) for j in range(N_anc)], dtype=complex) / np.sqrt(N_anc)
    
    # Aplicar IQFT
    IQFT = qft_matrix(n_ancilla).conj().T
    psi_final = IQFT @ psi_pre_iqft
    
    probs = np.abs(psi_final)**2
    return probs, phi

# Test con U = Rz(2π·0.375) → φ = 0.375 = 3/8
phi_test = 0.375
U_test = np.array([[np.exp(-1j*np.pi*phi_test), 0], [0, np.exp(1j*np.pi*phi_test)]], dtype=complex)
eigenstate_test = np.array([0, 1], dtype=complex)  # |1⟩ tiene eigenvalue e^(2πiφ)

for n_anc in [3, 4, 5]:
    probs, phi_real = qpe_matrix(U_test, eigenstate_test, n_anc)
    estado_max = np.argmax(probs)
    phi_estimada = estado_max / 2**n_anc
    print(f'n_ancilla={n_anc}: φ_real={phi_real:.6f}, estado_max={estado_max} ({estado_max:0{n_anc}b}), '
          f'φ_estimada={phi_estimada:.6f}, P(max)={probs[estado_max]:.4f}')

## Ejercicio 9.2 — Algoritmo de continuación de fracciones
QPE da una aproximación binaria de φ. Para Shor necesitamos recuperar el período r exacto.

In [ ]:
def continua_a_fraccion(phi, max_denominador=100):
    """Convierte una fracción continua en la fracción p/q más cercana."""
    frac = Fraction(phi).limit_denominator(max_denominador)
    return frac.numerator, frac.denominator

# Ejemplos relevantes para Shor
ejemplos = [
    (0b01100, 5, 'φ=12/32=0.375'),  # 3 bits: 011 → 3/8, pero con más ancilla
    (3/8,     8, 'φ=3/8 exacto'),
    (1/4,     8, 'φ=1/4'),
    (1/3,    10, 'φ≈1/3 (irracional → aproximar)'),
    (0.3333,  8, 'φ≈1/3 (desde binario ruidoso)'),
]

print(f'{"Descripción":35}  {"φ":10}  {"p/q":>10}  {"Período q":>10}')
print('-' * 75)
for phi_val, max_denom, desc in [(3/8, 8, 'φ=3/8'), (1/4, 8, 'φ=1/4'),
                                  (1/3, 10, 'φ≈1/3'), (0.3333, 8, '0.3333')]:
    p, q = continua_a_fraccion(phi_val, max_denom)
    print(f'{desc:20}  {phi_val:.6f}    {p}/{q:>5}       {q:>8}')

print()
print('En Shor: si QPE devuelve j/2^n ≈ k/r, entonces continua_a_fraccion recupera r.')

## Ejercicio 9.3 — Reducción de factorización a búsqueda de período
El núcleo matemático de Shor: factorizar N usando el período de f(x) = a^x mod N.

In [ ]:
def periodo_clasico(a, N):
    """Encuentra el período r de f(x) = a^x mod N clásicamente (para referencia)."""
    x = 1
    for r in range(1, N + 1):
        x = (x * a) % N
        if x == 1:
            return r
    return None

def shor_clasico(N):
    """Versión clásica del algoritmo de Shor para entender la estructura."""
    print(f'Factorizando N = {N}:')
    
    for intento in range(10):
        a = np.random.randint(2, N)
        g = gcd(a, N)
        
        if g != 1:
            print(f'  Suerte: gcd({a}, {N}) = {g} → factores: {g} × {N//g}')
            return g, N // g
        
        r = periodo_clasico(a, N)
        print(f'  Intento {intento+1}: a={a}, período r={r}', end='')
        
        if r is None or r % 2 == 1:
            print(' (período inútil, reintentar)')
            continue
        
        if pow(a, r//2, N) == N - 1:
            print(' (a^(r/2) ≡ -1 mod N, reintentar)')
            continue
        
        factor1 = gcd(pow(a, r//2) - 1, N)
        factor2 = gcd(pow(a, r//2) + 1, N)
        
        if factor1 not in [1, N] and factor2 not in [1, N]:
            print(f' → ÉXITO: {N} = {factor1} × {factor2}')
            return factor1, factor2
        else:
            print(' (factores triviales, reintentar)')
    
    return None

for N_test in [15, 21, 35, 77]:
    shor_clasico(N_test)
    print()

## Ejercicio 9.4 — Shor cuántico para N=15 (circuito simplificado)
Factorizamos 15 = 3 × 5 usando a=7. El período es r=4 (7⁴ ≡ 1 mod 15).

In [ ]:
# Verificar: 7^x mod 15
a, N = 7, 15
print(f'f(x) = {a}^x mod {N}:')
for x in range(9):
    print(f'  f({x}) = {pow(a, x, N)}')

r = periodo_clasico(a, N)
print(f'\nPeríodo r = {r}')
print(f'Factores: gcd({a}^(r/2)-1, {N}) = gcd({pow(a,r//2)-1}, {N}) = {gcd(pow(a,r//2)-1, N)}')
print(f'          gcd({a}^(r/2)+1, {N}) = gcd({pow(a,r//2)+1}, {N}) = {gcd(pow(a,r//2)+1, N)}')

print()
# Circuito Qiskit simplificado (oráculo hardcodeado para a=7, N=15)
# Ver Qiskit Textbook para la implementación completa del oráculo modular
print('Para N=15 con a=7, el circuito completo requiere ~20 qubits.')
print('Qiskit incluye una implementación completa en qiskit.algorithms.Shor')
print('que usaremos en la Semana 11 con hardware real.')

# Simular la distribución de probabilidades que produciría QPE
n_anc = 4  # 4 bits de ancilla para período r=4
# Las fases que debería medir: k·2^n_anc/r = k·16/4 = 0, 4, 8, 12
estados_esperados = [k * 2**n_anc // r for k in range(r)]
print(f'\nEstados que QPE debe medir (fases k/r): {estados_esperados}')
print('En binario:', [f'{s:04b}' for s in estados_esperados])
print('→ Fracción continua recupera el denominador 4 = período r')
for s in estados_esperados:
    phi = s / 2**n_anc
    _, q = continua_a_fraccion(phi, N)
    print(f'  estado {s:04b} ({s}) → φ={phi:.4f} → denominador={q}')

## Mini reto — Completar antes de avanzar a la Semana 10

Implementa QPE completo en Qiskit para la puerta T (fase π/4):
1. U = T, eigenstate = |1⟩, fase φ = 1/8
2. Usa n_ancilla = 4 (suficiente para representar 1/8 = 0.0010 en binario)
3. Verifica que el resultado más probable es el estado |0010⟩ = 2
4. Calcula φ_estimada = 2/16 = 1/8 ✓

In [ ]:
# TU CÓDIGO AQUÍ
def qpe_puerta_T() -> dict:
    """QPE para la puerta T (φ=1/8) con 4 ancilla."""
    # TODO: implementar circuito Qiskit
    pass

## ✅ Hito de la Semana 9

Antes de avanzar, comprueba que puedes:

- [ ] Implementar QPE matricialmente y verificar la precisión con n ancilla
- [ ] Usar fracciones continuas para recuperar el período desde una fase
- [ ] Ejecutar Shor clásico y entender cada paso de la reducción
- [ ] Calcular los estados que QPE debe medir para N=15, a=7
- [ ] Explicar por qué Shor es exponencialmente más rápido que GNFS clásico

## Reflexión (escribe aquí tu respuesta)

**¿Por qué el algoritmo de Shor tiene impacto en criptografía RSA?**

*(tu respuesta aquí)*

**¿Qué tamaño de computador cuántico se necesita para romper RSA-2048?**

*(tu respuesta aquí)*

---
*Siguiente: Semana 10 — Ruido, Decoherencia y Corrección de Errores*